In [11]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from imblearn.over_sampling import SMOTE




# Modelling experiment

This notebook includes a modelling exercise using the data with a dataset that includes only the considered relevant features:

## Dataset summary

Shape: 2686 rows × 49 columns.

Target Variable: inhospital_mortality (binary: 0 = survived, 1 = died).

Feature Types:
Numerical: Several features (e.g., agecalc_adm, height_cm_adm).

Categorical: Many features (e.g., sex_adm, oxygenavail_adm).

Missing Values: No missing values.

## Load the data

In [9]:
relevant_dataset = pd.read_csv('../../filtered_relevant_dataset.csv')

In [10]:
relevant_dataset

,agecalc_adm,height_cm_adm,weight_kg_adm,muac_mm_adm,hr_bpm_adm,rr_brpm_app_adm,sysbp_mmhg_adm,diasbp_mmhg_adm,temp_c_adm,spo2site1_pc_oxi_adm,...,tobacco_adm,bednet_adm,hematocrit_gpdl_adm,hctpretransfusion_adm,hivstatus_adm,lactate_mmolpl_adm,glucose_mmolpl_adm,sqi1_perc_oxi_adm,sqi2_perc_oxi_adm,inhospital_mortality
0,16.8,79.8,11.6,150.0,130.0,23.0,92.0,54.0,36.3,99.0,...,Never,Always,38.000000,Yes,HIV negative,2.2,4.8,98.0,97.0,0
1,46.1,93.0,13.6,151.0,115.0,35.0,72.0,42.0,36.8,92.0,...,Never,Always,42.000000,Yes,HIV negative,2.7,7.2,99.0,85.0,0
2,7.9,68.2,8.2,148.0,150.0,56.0,94.0,51.0,37.2,99.0,...,Never,Always,40.000000,Yes,HIV negative,2.9,5.0,98.0,97.0,0
3,38.2,95.0,12.0,138.0,134.0,38.0,94.0,57.0,37.6,98.0,...,Never,Always,33.853659,Yes,HIV negative,1.3,11.7,99.0,99.0,0
4,16.1,83.0,12.0,165.0,163.0,61.0,107.0,73.0,38.7,90.0,...,Never,Sometimes,42.000000,Yes,HIV negative,4.4,5.8,96.0,46.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2681,20.1,98.2,10.0,135.0,163.0,61.0,103.0,54.0,39.6,98.0,...,Never,Always,24.000000,Yes,HIV negative,0.9,4.3,93.0,91.0,0
2682,26.8,82.3,10.0,140.0,115.0,37.0,83.0,50.0,37.1,96.0,...,Never,Sometimes,31.960100,Yes,HIV negative,1.7,5.4,99.0,67.0,0
2683,14.6,82.4,12.7,145.0,150.0,40.0,93.0,35.0,38.5,98.0,...,Never,Always,46.000000,Yes,HIV negative,1.8,5.8,99.0,99.0,0
2684,9.5,67.5,7.5,130.0,145.0,26.0,88.0,52.0,37.3,99.0,...,Never,Always,40.000000,Yes,HIV negative,0.9,3.8,82.0,25.0,0


### List of Columns in the Filtered Dataset

1. `agecalc_adm`
2. `height_cm_adm`
3. `weight_kg_adm`
4. `muac_mm_adm`
5. `hr_bpm_adm`
6. `rr_brpm_app_adm`
7. `sysbp_mmhg_adm`
8. `diasbp_mmhg_adm`
9. `temp_c_adm`
10. `spo2site1_pc_oxi_adm`
11. `spo2site2_pc_oxi_adm`
12. `spo2onoxy_adm`
13. `oxygenavail_adm`
14. `respdistress_adm`
15. `caprefill_adm`
16. `bcseye_adm`
17. `bcsmotor_adm`
18. `bcsverbal_adm`
19. `bcgscar_adm`
20. `vaccpneumoc_adm`
21. `vaccdpt_adm`
22. `priorhosp_adm`
23. `prioryearcough_adm`
24. `diarrheaoften_adm`
25. `feedingstatus_adm`
26. `exclbreastfed_adm`
27. `totalbreastfed_adm`
28. `deliveryloc_adm`
29. `birthattend_adm`
30. `travelmethod_adm`
31. `traveldist_adm`
32. `badhealthduration_adm`
33. `caregiverage_adm`
34. `caregivermarried_adm`
35. `momageknown_adm`
36. `momhiv_adm`
37. `watersource_adm`
38. `waterpure_adm`
39. `lightfuel_adm`
40. `tobacco_adm`
41. `bednet_adm`
42. `hematocrit_gpdl_adm`
43. `hctpretransfusion_adm`
44. `hivstatus_adm`
45. `lactate_mmolpl_adm`
46. `glucose_mmolpl_adm`
47. `sqi1_perc_oxi_adm`
48. `sqi2_perc_oxi_adm`
49. `inhospital_mortality`


## Preprocessing

In [13]:
# Identify categorical columns (dtype 'object')
categorical_columns = relevant_dataset.select_dtypes(include=['object']).columns.tolist()

# Initialize OneHotEncoder
encoder = OneHotEncoder(drop='first', sparse=False)

# Apply OneHotEncoding to categorical columns
encoded_data = encoder.fit_transform(relevant_dataset[categorical_columns])

# Convert encoded data to a DataFrame
encoded_columns = encoder.get_feature_names_out(categorical_columns)
encoded_df = pd.DataFrame(encoded_data, columns=encoded_columns, index=relevant_dataset.index)

# Drop original categorical columns from the dataset
filtered_dataset_encoded = relevant_dataset.drop(columns=categorical_columns)

# Concatenate the encoded DataFrame with the rest of the dataset
final_encoded_dataset = pd.concat([filtered_dataset_encoded, encoded_df], axis=1)


In [14]:
final_encoded_dataset

,agecalc_adm,height_cm_adm,weight_kg_adm,muac_mm_adm,hr_bpm_adm,rr_brpm_app_adm,sysbp_mmhg_adm,diasbp_mmhg_adm,temp_c_adm,spo2site1_pc_oxi_adm,...,lightfuel_adm_Solar powered bulbs,lightfuel_adm_Tadooba,tobacco_adm_Less than monthly,tobacco_adm_Monthly,tobacco_adm_Never,tobacco_adm_Weekly,bednet_adm_Never,bednet_adm_Sometimes,hctpretransfusion_adm_Yes,hivstatus_adm_HIV positive
0,16.8,79.8,11.6,150.0,130.0,23.0,92.0,54.0,36.3,99.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
1,46.1,93.0,13.6,151.0,115.0,35.0,72.0,42.0,36.8,92.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2,7.9,68.2,8.2,148.0,150.0,56.0,94.0,51.0,37.2,99.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
3,38.2,95.0,12.0,138.0,134.0,38.0,94.0,57.0,37.6,98.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
4,16.1,83.0,12.0,165.0,163.0,61.0,107.0,73.0,38.7,90.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2681,20.1,98.2,10.0,135.0,163.0,61.0,103.0,54.0,39.6,98.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2682,26.8,82.3,10.0,140.0,115.0,37.0,83.0,50.0,37.1,96.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
2683,14.6,82.4,12.7,145.0,150.0,40.0,93.0,35.0,38.5,98.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2684,9.5,67.5,7.5,130.0,145.0,26.0,88.0,52.0,37.3,99.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


## Scaling

Since Standardization works well for Logistic Regression Models, I'll use this technique in this experiment.

In [18]:
# Identify numerical columns in the relevant dataset
numerical_columns = relevant_dataset.select_dtypes(include=['float64', 'int64']).columns.tolist()
numerical_columns.remove('inhospital_mortality')  # Exclude the target column

# Apply standardization to the numerical columns
scaler = StandardScaler()
scaled_numerical_data = scaler.fit_transform(relevant_dataset[numerical_columns])

# Convert scaled data back into a DataFrame
scaled_numerical_df = pd.DataFrame(scaled_numerical_data, columns=numerical_columns, index=relevant_dataset.index)

# Extract encoded categorical columns from the final encoded dataset
encoded_categorical_columns = final_encoded_dataset.drop(columns=numerical_columns + ['inhospital_mortality'])

# Combine scaled numerical data, encoded categorical data, and target column
final_combined_dataset = pd.concat(
    [scaled_numerical_df, encoded_categorical_columns, relevant_dataset['inhospital_mortality']],
    axis=1
)

In [19]:
final_combined_dataset

,agecalc_adm,height_cm_adm,weight_kg_adm,muac_mm_adm,hr_bpm_adm,rr_brpm_app_adm,sysbp_mmhg_adm,diasbp_mmhg_adm,temp_c_adm,spo2site1_pc_oxi_adm,...,lightfuel_adm_Tadooba,tobacco_adm_Less than monthly,tobacco_adm_Monthly,tobacco_adm_Never,tobacco_adm_Weekly,bednet_adm_Never,bednet_adm_Sometimes,hctpretransfusion_adm_Yes,hivstatus_adm_HIV positive,inhospital_mortality
0,-0.331171,0.011930,0.723506,0.748700,-0.439264,-1.560085,-0.416261,-0.053944,-1.161617,0.664525,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0
1,1.913211,1.201135,1.439671,0.811594,-1.026553,-0.752358,-2.101129,-1.144732,-0.665275,-0.431667,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0
2,-1.012912,-1.033129,-0.493975,0.622912,0.343788,0.661165,-0.247775,-0.326641,-0.268202,0.664525,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0
3,1.308070,1.381318,0.866739,-0.006030,-0.282654,-0.550426,-0.247775,0.218753,0.128871,0.507926,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0
4,-0.384791,0.300222,0.866739,1.692113,0.852771,0.997718,0.847389,1.673138,1.220823,-0.744865,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2681,-0.078391,1.669610,0.150574,-0.194712,0.852771,0.997718,0.510416,-0.053944,2.114238,0.507926,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0
2682,0.434829,0.237158,0.150574,0.119759,-1.026553,-0.617737,-1.174452,-0.417540,-0.367470,0.194728,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0
2683,-0.499691,0.246167,1.117397,0.434230,0.343788,-0.415805,-0.332018,-1.781025,1.022286,0.507926,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0
2684,-0.890352,-1.096193,-0.744633,-0.509183,0.148025,-1.358154,-0.753235,-0.235742,-0.168934,0.664525,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0


## Baseline Model

In [20]:
# Step 1: Split the data
X = final_combined_dataset.drop(columns=['inhospital_mortality'])
y = final_combined_dataset['inhospital_mortality']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 2: Initialize and train the Logistic Regression model
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train, y_train)

# Step 3: Make predictions
y_pred = log_reg.predict(X_test)
y_pred_prob = log_reg.predict_proba(X_test)[:, 1]  # Probability for the positive class

# Step 4: Evaluate the model
print("Classification Report:")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_pred_prob):.2f}")


Classification Report:
              precision    recall  f1-score   support

           0       0.94      1.00      0.97       507
           1       1.00      0.03      0.06        31

    accuracy                           0.94       538
   macro avg       0.97      0.52      0.52       538
weighted avg       0.95      0.94      0.92       538

ROC-AUC Score: 0.71


The class imbalance makes the model predicts the majority class (0) very well, but struggles with the minority class (1)..

## Class Imbalance

### Class Weighting 

In [23]:
# Initialize the Logistic Regression model with class weighting
log_reg_balanced = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)

# Train the model
log_reg_balanced.fit(X_train, y_train)

# Make predictions
y_pred_balanced = log_reg_balanced.predict(X_test)
y_pred_prob_balanced = log_reg_balanced.predict_proba(X_test)[:, 1]

# Evaluate the model
print("Classification Report with Class Weighting:")
print(classification_report(y_test, y_pred_balanced))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_pred_prob_balanced):.2f}")


Classification Report with Class Weighting:
              precision    recall  f1-score   support

           0       0.95      0.82      0.88       507
           1       0.11      0.35      0.17        31

    accuracy                           0.80       538
   macro avg       0.53      0.59      0.53       538
weighted avg       0.91      0.80      0.84       538

ROC-AUC Score: 0.66


Recall: Improved significantly to 35% (from 3% without weighting). The model identifies more true positive cases for the minority class.


Precision: Dropped to 11%, meaning many predicted positive cases are false positives.


F1-Score: 0.17, which reflects the balance between precision and recall but remains low due to poor precision.

### SMOTE

In [24]:
# Step 1: Split the data into train and test sets
X = final_combined_dataset.drop(columns=['inhospital_mortality'])
y = final_combined_dataset['inhospital_mortality']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 2: Apply SMOTE to the training set
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# Step 3: Train Logistic Regression on the SMOTE data
log_reg_smote = LogisticRegression(max_iter=1000, random_state=42)
log_reg_smote.fit(X_train_smote, y_train_smote)

# Step 4: Make predictions
y_pred_smote = log_reg_smote.predict(X_test)
y_pred_prob_smote = log_reg_smote.predict_proba(X_test)[:, 1]

# Step 5: Evaluate the model
print("Classification Report with SMOTE:")
print(classification_report(y_test, y_pred_smote))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_pred_prob_smote):.2f}")


Classification Report with SMOTE:
              precision    recall  f1-score   support

           0       0.96      0.83      0.89       507
           1       0.12      0.39      0.18        31

    accuracy                           0.80       538
   macro avg       0.54      0.61      0.54       538
weighted avg       0.91      0.80      0.85       538

ROC-AUC Score: 0.63


Improvement in Recall for Minority Class: SMOTE is effective in increasing recall, but precision suffers.

Trade-offs:
Precision drops due to oversampling introducing synthetic data.
ROC-AUC is slightly lower than the class-weighted approach.

### TO DO: 
    
    Explore the Feature Importance and Interpretability. 
    Run a Decision Tree Model
    Check again the models my team has run so far.
    Check the process to submit the first challenge entry
